In [1]:
import re
import math
import json

# ==============================================================================
# 1. CONFIGURATION & FILE PATHS
# ==============================================================================
PLAYERS_FILE = 'src/data/fortnite/players.ts'
ENTRIES_FILE = 'src/data/fortnite/entries.ts'
EVENTS_FILE  = 'src/data/fortnite/events.ts'

# Balanced 50/50 split between career financial dominance & prestige victories
EARNINGS_WEIGHT = 0.70
WINS_WEIGHT     = 0.30

EASY_PERCENTILE   = 0.10  # Top 10%
MEDIUM_PERCENTILE = 0.40  # 10% - 40%
# Remaining 40% - 100% is Hard

In [2]:
# ==============================================================================
# 2. EVENT WEIGHTING LOGIC
# ==============================================================================
def score_event(event: dict) -> float:
    eid = event.get('id', '')
    tier = event.get('tier', '')
    region = event.get('region')
    platform = event.get('platform')

    # Tier S+: Fortnite World Cup Solo / Duo
    if 'world-cup' in eid:
        return 100.0

    # Tier S: Global Championships & Official Global LANs
    if tier == 'global':
        return 70.0

    # LAN Majors (Esports World Cup, Gamers8, DreamHack)
    if tier == 'lan':
        if any(k in eid for k in ['gamers8', 'esports-world-cup', 'reload-elite']):
            return 15.0
        if 'dreamhack' in eid:
            return 15.0
        return 15.0

    # Tier D: Console & Mobile FNCS Finals
    if platform == 'Console/mobile':
        return 2.0

    # Regional Online FNCS
    if tier == 'fncs':
        if region == 'EU':
            return 30.0
        if region in ('NAC', 'NAE'):
            return 22.0
        if region in ('NAW', 'BR'):
            return 15.0
        if region in ('ASIA', 'ME', 'OCE'):
            return 8.0
        return 12.0

    # Legacy Majors: Summer/Fall Skirmish, Winter Royale
    if tier == 'major':
        return 12.0

    return 5.0


In [3]:
# ==============================================================================
# 3. PARSERS
# ==============================================================================
def load_events(path: str) -> dict:
    events = {}
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if "id:" in line and "tier:" in line:
                eid_match = re.search(r"id:\s*'([^']+)'", line)
                name_match = re.search(r"name:\s*'([^']+)'", line)
                short_match = re.search(r"shortName:\s*'([^']+)'", line)
                tier_match = re.search(r"tier:\s*'([^']+)'", line)
                region_match = re.search(r"region:\s*(?:'([^']+)'|null)", line)
                platform_match = re.search(r"platform:\s*(?:'([^']+)'|null)", line)

                if eid_match:
                    eid = eid_match.group(1)
                    evt = {
                        'id': eid,
                        'name': name_match.group(1) if name_match else eid,
                        'shortName': short_match.group(1) if short_match else eid,
                        'tier': tier_match.group(1) if tier_match else 'fncs',
                        'region': region_match.group(1) if region_match else None,
                        'platform': platform_match.group(1) if platform_match else None
                    }
                    evt['points'] = score_event(evt)
                    events[eid] = evt
    return events

def load_players(path: str) -> dict:
    players = {}
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if "id:" in line and "name:" in line:
                pid_match = re.search(r"id:\s*'([^']+)'", line)
                name_match = re.search(r"name:\s*'([^']+)'", line)
                earnings_match = re.search(r"earnings:\s*(\d+)", line)
                known_match = re.search(r"earningsKnown:\s*(true|false)", line)

                if pid_match and name_match:
                    pid = pid_match.group(1)
                    players[pid] = {
                        'id': pid,
                        'name': name_match.group(1),
                        'earnings': int(earnings_match.group(1)) if earnings_match else 0,
                        'earnings_known': (known_match.group(1) == 'true') if known_match else False,
                        'win_points': 0.0,
                        'major_wins': 0,
                        'win_titles': []
                    }
    return players

def load_entries(path: str, events: dict, players: dict):
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if "eventId:" in line and "placement:" in line:
                placement_match = re.search(r"placement:\s*(\d+)", line)
                if placement_match and int(placement_match.group(1)) == 1:
                    eid_match = re.search(r"eventId:\s*'([^']+)'", line)
                    pids_match = re.search(r"playerIds:\s*\[(.*?)\]", line)

                    if eid_match and pids_match:
                        eid = eid_match.group(1)
                        event = events.get(eid, {'points': 5.0, 'shortName': eid})
                        pids = re.findall(r"'([^']+)'", pids_match.group(1))

                        for pid in pids:
                            if pid in players:
                                players[pid]['major_wins'] += 1
                                players[pid]['win_points'] += event['points']
                                players[pid]['win_titles'].append(event['shortName'])


In [4]:
# 4. FAME CALCULATION
# ==============================================================================
def compute_fame_ranking(players: dict) -> list:
    # 1. Earnings Score (Log10 Normalization)
    known_log_earnings = [
        math.log10(p['earnings'])
        for p in players.values()
        if p['earnings_known'] and p['earnings'] > 0
    ]

    max_log_e = max(known_log_earnings) if known_log_earnings else 1.0
    min_log_e = min(known_log_earnings) if known_log_earnings else 0.0

    # Assign 25th-percentile log-earnings to unknown players
    sorted_known = sorted(known_log_earnings)
    baseline_log = sorted_known[int(len(sorted_known) * 0.25)] if sorted_known else 0.0

    for p in players.values():
        if p['earnings_known'] and p['earnings'] > 0:
            val = math.log10(p['earnings'])
        else:
            val = baseline_log

        p['earnings_score'] = (val - min_log_e) / (max_log_e - min_log_e) if max_log_e > min_log_e else 0.0

    # 2. Wins Score (Normalized by Max Win Points in DB)
    max_pts = max([p['win_points'] for p in players.values()]) if players else 1.0
    for p in players.values():
        p['wins_score'] = p['win_points'] / max_pts if max_pts > 0 else 0.0

    # 3. Composite Fame Score
    for p in players.values():
        raw_fame = (EARNINGS_WEIGHT * p['earnings_score']) + (WINS_WEIGHT * p['wins_score'])
        p['fame_score'] = raw_fame * 100.0

    # Return descending sorted list
    return sorted(players.values(), key=lambda x: x['fame_score'], reverse=True)

In [5]:
# ==============================================================================
# 5. REPORT & JSON EXPORT
# ==============================================================================
def display_report(sorted_players: list):
    total = len(sorted_players)
    easy_cutoff = int(total * EASY_PERCENTILE)
    medium_cutoff = int(total * MEDIUM_PERCENTILE)

    print("=" * 80)
    print(f"TIER CUTOFFS (Total Players: {total})")
    print(f"  ⭐ Easy   (Top {int(EASY_PERCENTILE*100)}%):   #1 – #{easy_cutoff}")
    print(f"  🟡 Medium ({int(EASY_PERCENTILE*100)}–{int(MEDIUM_PERCENTILE*100)}%):  #{easy_cutoff + 1} – #{medium_cutoff}")
    print(f"  🔴 Hard   ({int(MEDIUM_PERCENTILE*100)}–100%): #{medium_cutoff + 1} – #{total}")
    print(f"Algorithm Weights: {int(EARNINGS_WEIGHT*100)}% Log Earnings / {int(WINS_WEIGHT*100)}% Weighted Victories")
    print("=" * 80)

    # Print Top 50 Players for visual validation
    for i, p in enumerate(sorted_players[:50]):
        rank = i + 1
        tier_tag = "⭐ Easy" if rank <= easy_cutoff else "🟡 Medium" if rank <= medium_cutoff else "🔴 Hard"
        earnings_disp = f"${p['earnings']:,}" if p['earnings_known'] else "Unknown"
        titles_disp = ", ".join(p['win_titles']) if p['win_titles'] else "None"

        print(f"#{rank:<2} {p['name']:<16} [{tier_tag}]  Fame: {p['fame_score']:5.2f} pts")
        print(f"    Earnings: {earnings_disp:<12} (LogScore: {p['earnings_score']:.2f})")
        print(f"    Prestige: {p['win_points']:4.0f} pts       (WinScore: {p['wins_score']:.2f}, {p['major_wins']} wins)")
        print(f"    Titles:   {titles_disp}")
        print("-" * 80)

    # Export to JSON
    export_payload = [
        {
            "playerId": p['id'],
            "name": p['name'],
            "fameScore": round(p['fame_score'] / 100.0, 4),
            "famePercentile": round((i + 1) / total, 4),
            "tier": "easy" if (i + 1) <= easy_cutoff else "medium" if (i + 1) <= medium_cutoff else "hard",
            "prestigePoints": p['win_points']
        }
        for i, p in enumerate(sorted_players)
    ]

    with open("fame-ranking.json", "w", encoding='utf-8') as f:
        json.dump(export_payload, f, indent=2)
    print("\nSaved full ranking payload to 'fame-ranking.json'.")


In [6]:
if __name__ == "__main__":
    events = load_events(EVENTS_FILE)
    players = load_players(PLAYERS_FILE)
    load_entries(ENTRIES_FILE, events, players)
    ranked = compute_fame_ranking(players)
    display_report(ranked)

TIER CUTOFFS (Total Players: 316)
  ⭐ Easy   (Top 10%):   #1 – #31
  🟡 Medium (10–40%):  #32 – #126
  🔴 Hard   (40–100%): #127 – #316
Algorithm Weights: 70% Log Earnings / 30% Weighted Victories
#1  Bugha            [⭐ Easy]  Fame: 90.33 pts
    Earnings: $3,844,147   (LogScore: 1.00)
    Prestige:  166 pts       (WinScore: 0.68, 4 wins)
    Titles:   FNCS: Chapter 2 Season 8, 2021 FNCS Grand Royale, FNCS: Chapter 3 Season 1, Fortnite World Cup (Solos)
--------------------------------------------------------------------------------
#2  Malibuca         [⭐ Easy]  Fame: 83.49 pts
    Earnings: $1,562,371   (LogScore: 0.76)
    Prestige:  245 pts       (WinScore: 1.00, 6 wins)
    Titles:   FNCS Major 1 – 2024, FNCS Major 1 Summit – 2026, Gamers8 2022 (Battle Royale), DreamHack Dallas 2023, DreamHack Summer 2023, 2026 Reload Elite Series Championship (2026 Esports World Cup)
--------------------------------------------------------------------------------
#3  Aqua             [⭐ Easy]  Fam